# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [28]:
# Import the standard library module used to read environment variables.

import os

# Import load_dotenv so the notebook can load secrets from a local .env file.
from dotenv import load_dotenv

# Import IPython display helpers so streamed text can be rendered as Markdown in the notebook.
from IPython.display import Markdown, display, update_display

# Import the OpenAI client. The same client class is used for both OpenAI and Ollama-compatible APIs.
from openai import OpenAI


In [29]:
# Define the hosted OpenAI model used by default.

MODEL_GPT = 'gpt-4o-mini'

# Define the local Ollama model name used when running against Ollama.
MODEL_LLAMA = 'llama3.2'

# Ollama exposes an OpenAI-compatible API at this local base URL.
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Create a client pointed at Ollama. Ollama does not need a real API key, but the client requires a value.
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# Load the environment variables needed for the hosted OpenAI API.

# Read key-value pairs from a local .env file, replacing any existing values in this session.
load_dotenv(override=True)

# Pull the OpenAI API key out of the environment so we can validate it before creating requests.
api_key = os.getenv('OPENAI_API_KEY')

# If no key is available, print a troubleshooting hint instead of failing with a less helpful API error later.
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# If the key does not look like a project key, warn the user that it may be the wrong credential type.
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# If the key has leading or trailing whitespace, warn the user because that commonly breaks authentication.
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# If the key passes the basic checks, confirm that the notebook is ready to call the hosted OpenAI API.
else:
    print("API key found and looks good so far!")


# Create the hosted OpenAI client. It automatically reads OPENAI_API_KEY from the environment.
openai = OpenAI()


In [ ]:
# Build the prompt and helper functions used to answer technical questions.

# The system prompt defines the assistant's role and the style of explanation we want.
system_prompt = """
You are a technical tutor who breaks down complex programming concepts into simple, easy-to-understand explanations.  
If the question is extremely simple, assume the user may need foundational concepts explained first.
"""

# This prefix is added before the user's question so the model knows the exact task.
user_prompt_prefix = """
Please explain what this code does and why:
"""

# Convert one plain-text question into the chat message format expected by the OpenAI API.
def messages_for(question):
    # Send the tutor instructions first, then send the user's actual question.
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + question}
    ]


# Ask either the hosted OpenAI model or the local Ollama model, then stream the response into the notebook.
def stream_answer(question, model=MODEL_GPT):
    # Use the hosted OpenAI client when the selected model is the default GPT model.
    if model == MODEL_GPT:
        stream = openai.chat.completions.create(
            model=model,
            messages=messages_for(question),
            stream=True
        )    
    # Use the local Ollama client when the selected model is the Llama model.
    elif model == MODEL_LLAMA:
        stream = ollama.chat.completions.create(
            model=model,
            messages=messages_for(question),
            stream=True
        )
    # Fail clearly if a caller passes a model this helper does not know how to route.
    else:
        raise ValueError(f"Unknown model: {model}")

    # Accumulate the streamed text because each chunk contains only the newest fragment.
    response = ""

    # Create an empty Markdown display area that will be updated in place as tokens arrive.
    display_handle = display(Markdown(""), display_id=True)

    # Process each streamed chunk from the model.
    for chunk in stream:
        # Add the latest text fragment, using an empty string when a chunk contains no content.
        response += chunk.choices[0].delta.content or ''
        # Re-render the full response so the notebook shows a live, growing answer.
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# Ask the default hosted GPT model to answer a sample technical question, streaming the result as it arrives.

stream_answer("what does the os module do in python?")

In [ ]:
# Ask the local Llama 3.2 model the same question so the hosted and local responses can be compared.

stream_answer("what does the os module do in python?", MODEL_LLAMA)